# Chapitre 9 - Cours 4 : le consensus et la diffusion à ordre total

Élire un leader sans cerveau divisé, attribuer un nom d'utilisateur une seule fois,
s'accorder sur l'ordre des écritures d'un journal de réplication : tout cela est facile
sur **un** nœud et difficile dès qu'on veut **tolérer les pannes**. Ce notebook montre que
ces problèmes sont des formes d'un même problème, le **consensus** : il le définit,
l'implémente (Paxos à une décision) et établit l'équivalence entre consensus,
**diffusion à ordre total** et **registre linéarisable avec compare-and-set**. Il couvre
la fin de la section *Ordering Guarantees* (*Total Order Broadcast* : *Using total order
broadcast*, *Implementing linearizable storage using total order broadcast*,
*Implementing total order broadcast using linearizable storage*) et, dans la section
*Distributed Transactions and Consensus*, l'introduction et le début de *Fault-Tolerant
Consensus* (encadré *The Impossibility of Consensus*). La validation à deux phases a été
vue au cours 3 ; le cours 5 implémente Raft.

**Plan.**

1. Pourquoi le consensus ?
2. Définition : accord, intégrité, validité, terminaison.
3. L'impossibilité FLP, et pourquoi le consensus marche quand même.
4. Le journal partagé (diffusion à ordre total).
5. Compare-and-set et consensus sont équivalents.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ddia import consistency as cs
from ddia.diagrams import Diagram
from ddia.plotting import (AMBER, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET,
                           annotate_bars, set_style, subtitle)
from ddia.timelines import SpaceTime

set_style()
plt.rcParams["axes.titlepad"] = 16
pd.set_option("display.precision", 3)

## 1. Pourquoi le consensus ? (section *Distributed Transactions and Consensus*)

Les cours précédents ont accumulé des choses **faciles sur un seul nœud** et difficiles à
rendre tolérantes aux pannes :

| Sur un nœud | Le problème dès qu'on veut tolérer une panne |
|---|---|
| un leader unique, linéarisable | comment basculer sans cerveau divisé ? comment savoir qu'un leader suspendu n'a pas été remplacé ? |
| un compteur `fetch-and-add` qui distribue des identifiants | et s'il tombe en panne ? |
| un CAS atomique pour attribuer un verrou, un bail, un nom unique | une instruction du processeur ; comment la rendre tolérante ? |
| un journal de réplication écrit par le leader | qui décide de l'ordre des entrées si le leader meurt ? |
| valider une transaction sur plusieurs nœuds (cours 3) | tous les participants doivent prendre la même décision |

Toutes ces questions sont des instances d'un même problème fondamental de l'informatique
répartie : le **consensus**, faire **s'accorder plusieurs nœuds sur une valeur**. Il est
réputé difficile à réussir, et beaucoup de systèmes s'y sont trompés. Les algorithmes les
plus connus sont Viewstamped Replication, Paxos, Raft et Zab : proches, mais pas
identiques. Tous supposent un modèle **non byzantin** (les messages peuvent être perdus
ou retardés, les nœuds tomber en panne et redémarrer, mais aucun ne ment). Il existe des
algorithmes tolérant moins d'un tiers de nœuds byzantins (utilisés par les chaînes de
blocs, par exemple), hors du champ du livre (chapitre 8).

> **À retenir.** Leader, verrou, unicité, compteur, journal, validation atomique : le
> même problème, le consensus, se cache derrière chaque « facile sur un nœud ».